# Day 4 — Column Operations

**What you will learn:**
- Three ways to reference a column: `col()`, `df["col"]`, string
- `lit()` for literal values, `expr()` for SQL expressions
- `withColumn()`, `withColumnRenamed()`, `drop()`
- `cast()` for type conversions
- `when()` / `otherwise()` — the Spark equivalent of if/else
- `isNull()`, `isNotNull()`, `isin()`

**Exam relevance:** DataFrame API (30%) — column operations are the highest-density topic in the exam.

In [ ]:
from pyspark.sql.functions import col, lit, expr, when

data = [
    (1,  "Alice",   "Engineering", 95000,  None),
    (2,  "Bob",     "Marketing",   72000,  "Senior"),
    (3,  "Carol",   "Engineering", 110000, "Lead"),
    (4,  "Dave",    "Sales",        58000,  None),
    (5,  "Eve",     "Marketing",    81000,  "Senior"),
    (6,  "Frank",   "Engineering",  88000,  "Mid"),
]
df = spark.createDataFrame(data, ["id", "name", "dept", "salary", "level"])
df.show()

## 1. Three Ways to Reference a Column

All three are equivalent — but `col()` is the most portable (doesn't tie you to a specific DataFrame variable).

In [ ]:
# Option 1: col() from pyspark.sql.functions — preferred
df.select(col("name"), col("salary")).show(3)

# Option 2: df["column"] — ties to a specific DataFrame
df.select(df["name"], df["salary"]).show(3)

# Option 3: string name — only works with select, not in expressions
df.select("name", "salary").show(3)

## 2. lit() — Literal Values

Use `lit()` when you want to add a constant value as a column.

In [ ]:
from pyspark.sql.functions import lit

# Add a constant column — useful for tagging data with a source, version, or flag
df.withColumn("source", lit("HR_system")) \
  .withColumn("bonus_pct", lit(0.10)) \
  .select("name", "salary", "source", "bonus_pct") \
  .show()

## 3. expr() — SQL Expressions as Strings

`expr()` lets you write SQL syntax inside a DataFrame operation — useful for complex expressions.

In [ ]:
from pyspark.sql.functions import expr

df.withColumn("salary_after_tax", expr("salary * 0.78")) \
  .withColumn("name_dept", expr("concat(name, ' - ', dept)")) \
  .select("name", "salary", "salary_after_tax", "name_dept") \
  .show()

## 4. withColumn(), withColumnRenamed(), drop()

In [ ]:
# withColumn — add or OVERWRITE a column
df2 = df.withColumn("salary", col("salary") * 1.05)  # 5% raise — overwrites existing column
df2.select("name", "salary").show()

# withColumnRenamed — rename without touching data
df3 = df.withColumnRenamed("dept", "department").withColumnRenamed("level", "job_level")
print(df3.columns)

# drop — remove one or more columns
df4 = df.drop("id", "level")
print(df4.columns)

## 5. cast() — Type Conversions

> **Exam tip:** `cast()` returns `null` (not an error) if the conversion fails. Always verify after casting.

In [ ]:
from pyspark.sql.types import DoubleType, StringType, IntegerType

df.printSchema()  # salary is currently LongType (bigint)

# Cast using type string — concise
df_cast = df.withColumn("salary", col("salary").cast("double")) \
            .withColumn("id",     col("id").cast("string"))
df_cast.printSchema()

# Cast using type class — explicit
df_cast2 = df.withColumn("salary", col("salary").cast(DoubleType()))
df_cast2.printSchema()

## 6. when() / otherwise() — Conditional Columns

This is Spark's `CASE WHEN ... THEN ... ELSE ... END` from SQL.

In [ ]:
from pyspark.sql.functions import when

# Single condition
df.withColumn("is_senior", when(col("salary") >= 90000, True).otherwise(False)).show()

# Multiple conditions (chained when)
df.withColumn("band",
    when(col("salary") >= 100000, "Band A")
   .when(col("salary") >= 80000,  "Band B")
   .when(col("salary") >= 60000,  "Band C")
   .otherwise("Band D")
).select("name", "salary", "band").show()

## 7. isNull(), isNotNull(), isin()

In [ ]:
# isNull — rows where level is null
df.filter(col("level").isNull()).show()

# isNotNull — rows where level is populated
df.filter(col("level").isNotNull()).show()

# isin — rows matching a list of values
df.filter(col("dept").isin("Engineering", "Marketing")).show()

# Combine with when — fill nulls with a label
df.withColumn("level",
    when(col("level").isNull(), "Junior").otherwise(col("level"))
).show()

## 8. alias() — Rename Column in Select

`alias()` renames a column expression inline — useful when you want to name a computed column.

In [ ]:
df.select(
    col("name"),
    col("salary").alias("annual_salary"),
    (col("salary") / 12).alias("monthly_salary"),
    col("dept").alias("department")
).show()

---

## Day 4 Checklist

- [ ] Know the 3 ways to reference a column — and when to prefer `col()`
- [ ] Used `lit()` to add a constant column
- [ ] Used `expr()` to write SQL inside a DataFrame operation
- [ ] Used `withColumn()` to add and overwrite columns
- [ ] Used `withColumnRenamed()` and `drop()`
- [ ] Cast a column using both string shorthand and type class
- [ ] Built a multi-condition `when/otherwise` block
- [ ] Used `isNull()`, `isNotNull()`, `isin()`
- [ ] Used `alias()` to name computed columns

**Next:** Day 5 — Filtering, Sorting & Null Handling